# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook:** c16 — Timber Stock Dataset Generation  
**Author:** Jasper Cluistra  
**Last Updated:** 2026-05-17

---

### Goal
Generate the timber stock datasets used by the optimizer when assigning cross-sections to structural members. New stock is drawn from a parameterised catalogue covering standard softwood sections; reclaimed stock is sampled from a synthetic donor building.

### Inputs
- Stock generation parameters (`c16_generation_timber`)
- Representative beam length statistics (`representative_beam_statistics.json` from c12_15)

### Outputs
| File | Description |
|------|-------------|
| `new_timber_v2.csv` | New softwood stock only |
| `complete_timber_v2.csv` | New + reclaimed stock combined — used by the optimizer |
| `complete_timber_small.csv` | Small mixed subset for fast development runs |

# 1. Stock generation
Generates new and reclaimed stock, combines them, and saves the results to `config.TIMBER_STOCK_PATH`. Set `efficient = False` to use realistic (randomised) transport distances instead of the minimum-distance baseline.

In [ ]:
import pandas as pd
import config
from c16_generation_timber import generate_new_stock, generate_reclaimed_stock

efficient = True  # Set to True for efficient transport distances, False for realistic distances

# ── Generate New Stock ───────────────────────────────────────────────────────────────
df_new_stock = generate_new_stock(efficient)
display(df_new_stock.head(3))

# ── Generate Reclaimed Stock ───────────────────────────────────────────────────────────────
df_reclaimed_stock = generate_reclaimed_stock()
display(df_reclaimed_stock.head(3))

# ── Combine Datasets ───────────────────────────────────────────────────────────────────────
df_combined_stock = pd.concat([df_new_stock, df_reclaimed_stock], ignore_index=True)

# ── Save ───────────────────────────────────────────────────────────────────────
new_stock_dataset_filename = 'new_timber_v2.csv'
df_new_stock.to_csv(config.TIMBER_STOCK_PATH / new_stock_dataset_filename, index=False, sep=';')
combined_dataset_filename = 'complete_timber_v2.csv'
df_combined_stock.to_csv(config.TIMBER_STOCK_PATH / combined_dataset_filename, index=False, sep=';')

print(f"New stock dataset saved as '{new_stock_dataset_filename}'.")
print(f"Combined dataset saved as '{combined_dataset_filename}'.")

## 2. Section combinations
Inspect all unique Depth × Width combinations present in the new stock catalogue. Uncomment to run.

In [ ]:
# depth_width_combinations = (
#     df_new_stock[['Depth', 'Width']]
#     .drop_duplicates()
#     .sort_values(by=['Depth', 'Width'])
#     .reset_index(drop=True)
# )
#
# print(f"\n{'#':<6} {'D (mm)':<12} {'W (mm)':<12}")
# print("-" * 30)
# for idx, (_, row) in enumerate(depth_width_combinations.iterrows(), 1):
#     print(f"{idx:<6} {int(row['Depth']):<12} {int(row['Width']):<12}")
# print("-" * 30)
# print(f"Total combinations: {len(depth_width_combinations)}")

## 3. Development subset
Generates a small mixed stock file (`complete_timber_small.csv`) for fast development and testing runs in the optimizer. Adjust `total_elements` and `reclaimed_ratio` as needed.

In [ ]:
import config
from c16_generation_timber import generate_mixed_stock_subset

df_small = generate_mixed_stock_subset(
    total_elements   = 20,
    reclaimed_ratio  = 0.3,
    random_state     = 38,
    allow_replacement = True,
)

display(df_small.head(5))

dataset_filename = 'complete_timber_small.csv'
df_small.to_csv(config.TIMBER_STOCK_PATH / dataset_filename, index=False, sep=';')
print(f"Subset saved as '{dataset_filename}'  ({len(df_small)} elements).")